In [1]:
import h5py
import torch
import pickle 
from tqdm import tqdm
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Dataset
import plotly.graph_objects as go
from itertools import chain
import torch.nn.functional as F
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from sklearn.metrics import precision_score, recall_score, f1_score
import os


## Reading Data

In [2]:
#pickle_path = '/vast/da3245/datasets/stan_ds_wi_trace/traces.pkl'
pickle_path = '/vast/da3245/datasets/neuro_scans/training_set.pkl'
with open(pickle_path, 'rb') as f:
    trained_data = pickle.load(f) 

In [3]:
trained_data.keys()

dict_keys(['KG126', 'KG129', 'KG128', 'KG127', 'KG125'])

In [4]:
key = list(trained_data.keys())[0]

In [5]:
trained_data[key]['EXPT_1']['SCAN_98']['ROIS']['SOMA'].keys()

dict_keys(['PROCESSED', 'TRAINING_SET'])

In [6]:
trained_data[key]['EXPT_1']['SCAN_98']['ROIS']['SOMA']['TRAINING_SET'].keys()

dict_keys(['TRACE', 'FAST', 'SLOW'])

In [7]:
trained_data[key]['EXPT_1']['SCAN_98']['ROIS']['SOMA']['TRAINING_SET']['TRACE']

array([1.1469202 , 1.22046949, 1.19028212, ..., 0.5690665 , 0.55746104,
       0.5460146 ], shape=(96540,))

In [8]:
import numpy as np

# --- 1. Define Parameters ---
WINDOW_SIZE = 500
NUM_EVENTS = 2 # 'FAST' and 'SLOW'

# --- 2. Initialize Lists to Store Results ---
list_of_windowed_traces = []
list_of_encoded_labels = []
list_of_origin_keys = []

print("Processing data from all subjects, experiments, and scans...")
print("-" * 30)

# --- 3. Loop Through Each Layer of the Data ---
# Layer 1: Subjects (e.g., 'KG126', 'KG129', ...)
for subject_key in trained_data.keys():
    # Layer 2: Experiments for the current subject
    for expt_key in trained_data[subject_key].keys():
        # Layer 3: Scans for the current experiment
        for scan_key in trained_data[subject_key][expt_key].keys():
            
            # Create a unique ID to track the origin of the data
            origin_id = f"{subject_key}_{expt_key}_{scan_key}"
            print(f"Processing: {origin_id}")
            
            try:
                # Navigate the rest of the path to the actual data arrays
                data_path = trained_data[subject_key][expt_key][scan_key]['ROIS']['SOMA']['TRAINING_SET']

                trace_data = data_path['TRACE']
                fast_labels = data_path['FAST']
                slow_labels = data_path['SLOW']
                
                # Stack the two label types into one array for processing
                label_data = np.stack((fast_labels, slow_labels), axis=0)

            except KeyError:
                print(f"  -> Warning: Path not found or incomplete in '{origin_id}'. Skipping.")
                continue

            # --- (The rest of the processing logic is the same as before) ---

            # Calculate the number of full windows we can create
            num_windows = len(trace_data) // WINDOW_SIZE
            if num_windows == 0:
                print(f"  -> Warning: Not enough data in '{origin_id}' to create a window. Skipping.")
                continue

            # Truncate data to fit full windows
            truncate_len = num_windows * WINDOW_SIZE

            # Reshape the trace data into windows
            windowed_trace = trace_data[:truncate_len].reshape(num_windows, WINDOW_SIZE)

            # Reshape the stacked label data into windows
            windowed_labels = label_data[:, :truncate_len].reshape(NUM_EVENTS, num_windows, WINDOW_SIZE).transpose(1, 0, 2)

            # Create the encoded label for each window
            event_present = np.any(windowed_labels, axis=2)
            weights = 2**np.arange(NUM_EVENTS) # [1, 2]
            encoded_labels_for_scan = np.dot(event_present, weights)
            
            print(f"  -> Found {num_windows} windows.")

            # Add the results from this scan to our master lists
            list_of_windowed_traces.append(windowed_trace)
            list_of_encoded_labels.append(encoded_labels_for_scan)
            list_of_origin_keys.append(np.full(num_windows, origin_id))


# --- 4. Concatenate All Results into Single NumPy Arrays ---
# This check prevents an error if no data was processed at all
if list_of_windowed_traces:
    all_windowed_traces = np.concatenate(list_of_windowed_traces, axis=0)
    all_encoded_labels = np.concatenate(list_of_encoded_labels, axis=0)
    all_origin_keys = np.concatenate(list_of_origin_keys, axis=0)

    print("-" * 30)
    print("Processing Complete! ✅")
    print(f"Final shape of all windowed traces: {all_windowed_traces.shape}")
    print(f"Final shape of all encoded labels: {all_encoded_labels.shape}")
    print(f"Final shape of all origin keys:   {all_origin_keys.shape}")
    print("-" * 30)
    print("Label Encoding Scheme:")
    print("  0: No event")
    print("  1: FAST event only")
    print("  2: SLOW event only")
    print("  3: Both FAST and SLOW events")
    print("-" * 30)
    print("Example of first 5 labels:", all_encoded_labels[:5])
    print("Example of first 5 origin keys:", all_origin_keys[:5])
else:
    print("-" * 30)
    print("Processing Complete! ⚠️ No data was found or processed.")

Processing data from all subjects, experiments, and scans...
------------------------------
Processing: KG126_EXPT_3_SCAN_53
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_157
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_181
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_114
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_76
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_72
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_179
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_183
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_90
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_175
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_184
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_165
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_97
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_49
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_176
  -> Found 214 windows.
Processing: KG126_EXPT_3_SCAN_189
  -> Found

In [9]:
"""
window_processing.py  ▸  Version 2025‑07‑23
------------------------------------------------
Utility for slicing raw calcium‑trace recordings (or any 1‑D time‑series) into
fixed‑length windows, extracting per‑sample event labels (FAST, SLOW), and
producing three aligned outputs that downstream notebooks / Streamlit apps can
persist:

    • all_windowed_traces  –  (N, 500) float32
    • all_windowed_labels  –  (N, 2, 500) uint8   # FAST / SLOW indicators per sample
    • all_encoded_labels   –  (N,)     uint8      # 0=None, 1=FAST, 2=SLOW, 3=BOTH
    • all_origin_keys      –  (N,)     object     # e.g. "KG126_expt2_scan44"

The code mirrors your previous logic but now **retains `windowed_labels`**, so
you can save them to Zarr later.
"""

from __future__ import annotations

import numpy as np
from typing import Dict, Tuple, Any

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
WINDOW_SIZE: int = 500   # samples per window
NUM_EVENTS: int = 2      # FAST, SLOW (order matters!)

# ---------------------------------------------------------------------------
# Main helper
# ---------------------------------------------------------------------------

def extract_windows(
    trained_data: Dict[str, Any],
    window_size: int = WINDOW_SIZE,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Slice every TRACE in *trained_data* into fixed windows.

    Parameters
    ----------
    trained_data : nested dict
        Outer levels are **subject ➜ experiment ➜ scan** keys matching your raw
        pickle / MAT structure.  Leaf contains `ROIS/SOMA/TRAINING_SET` with
        arrays `TRACE`, `FAST`, `SLOW` (all 1‑D & length ≥ window_size).
    window_size : int, default 500
        Length of each window in samples.

    Returns
    -------
    all_windowed_traces : (N, window_size) float32
    all_windowed_labels : (N, NUM_EVENTS, window_size) uint8
    all_encoded_labels  : (N,) uint8
    all_origin_keys     : (N,) object
    """
    # Collect here, then concat once for speed.
    traces_list: list[np.ndarray]  = []
    labels_list: list[np.ndarray]  = []
    enc_lbl_list: list[np.ndarray] = []
    origin_list: list[np.ndarray]  = []

    weights = 2 ** np.arange(NUM_EVENTS, dtype=np.uint8)  # [1, 2]

    print("Processing data from all subjects, experiments, and scans…")
    print("-" * 30)

    for subj_key, subj_val in trained_data.items():
        for expt_key, expt_val in subj_val.items():
            for scan_key, scan_val in expt_val.items():
                origin_id = f"{subj_key}_{expt_key}_{scan_key}"
                print(f"Processing: {origin_id}")

                try:
                    ds = scan_val["ROIS"]["SOMA"]["TRAINING_SET"]
                    trace   = ds["TRACE"].astype(np.float32)
                    fast_lb = ds["FAST"].astype(np.uint8)
                    slow_lb = ds["SLOW"].astype(np.uint8)
                except KeyError:
                    print(f"  -> Warning: path missing in '{origin_id}'. Skipping.")
                    continue

                # stack labels into shape (2, L)
                label_stack = np.stack((fast_lb, slow_lb), axis=0)

                # How many full windows fit?
                num_win = trace.shape[0] // window_size
                if num_win == 0:
                    print(f"  -> Warning: <1 window in '{origin_id}'. Skipping.")
                    continue

                # trim to multiple of window_size
                end = num_win * window_size
                trace_w = trace[:end].reshape(num_win, window_size)
                labels_w = (
                    label_stack[:, :end]
                    .reshape(NUM_EVENTS, num_win, window_size)
                    .transpose(1, 0, 2)   # (num_win, 2, window)
                )

                # Encode per‑window event presence
                event_present = labels_w.any(axis=2)  # (num_win, 2) bool
                enc_labels = (event_present * weights).sum(axis=1).astype(np.uint8)

                print(f"  -> Stored {num_win} windows.")

                traces_list.append(trace_w)
                labels_list.append(labels_w)
                enc_lbl_list.append(enc_labels)
                origin_list.append(np.full(num_win, origin_id, dtype=object))

    # ---------------------------------------------------------------------
    # Concatenate & return
    # ---------------------------------------------------------------------
    if not traces_list:
        raise RuntimeError("No data processed – check input structure.")

    all_windowed_traces = np.concatenate(traces_list, axis=0)
    all_windowed_labels = np.concatenate(labels_list, axis=0)
    all_encoded_labels  = np.concatenate(enc_lbl_list, axis=0)
    all_origin_keys     = np.concatenate(origin_list, axis=0)

    # Nice summary
    print("-" * 30)
    print("Processing complete ✅")
    print(f"Traces  shape: {all_windowed_traces.shape}")
    print(f"Labels  shape: {all_windowed_labels.shape}")
    print(f"EncLbls shape: {all_encoded_labels.shape}")
    print(f"Origins shape: {all_origin_keys.shape}")
    print("-" * 30)
    print("Label encoding: 0=None, 1=FAST, 2=SLOW, 3=BOTH")
    print("Example enc  :", all_encoded_labels[:5])

    return (
        all_windowed_traces,
        all_windowed_labels,
        all_encoded_labels,
        all_origin_keys,
    )

# ---------------------------------------------------------------------------
# Example usage (inside your notebook)
# ---------------------------------------------------------------------------
out = extract_windows(trained_data)
traces_np, labels_np, enc_lbls_np, origins_np = out


Processing data from all subjects, experiments, and scans…
------------------------------
Processing: KG126_EXPT_3_SCAN_53
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_157
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_181
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_114
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_76
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_72
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_179
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_183
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_90
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_175
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_184
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_165
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_97
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_49
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_176
  -> Stored 214 windows.
Processing: KG126_EXPT_3_SCAN_1

In [10]:
[traces_np, labels_np, enc_lbls_np, origins_np]

[array([[ 0.09873113,  0.08969961,  0.08417262, ...,  0.27650648,
          0.27979603,  0.2816338 ],
        [ 0.28198838,  0.2808709 ,  0.27832922, ...,  0.05931573,
          0.07089526,  0.08427428],
        [ 0.09931671,  0.11585332,  0.13368674, ...,  0.23040198,
          0.22964318,  0.23055205],
        ...,
        [ 0.21804409,  0.2317286 ,  0.24319041, ...,  0.1863164 ,
          0.18178889,  0.17679729],
        [ 0.17104208,  0.16421907,  0.15603852, ...,  0.03562929,
          0.02988804,  0.02357833],
        [ 0.01653227,  0.00859915, -0.00034433, ..., -0.08906976,
         -0.09549021, -0.10135679]], shape=(32754, 500), dtype=float32),
 array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
     

In [11]:
np.array(enc_lbls_np).shape

(32754,)

In [12]:
np.array(origins_np).shape

(32754,)

In [13]:
windows = np.array(traces_np)
windows.shape

(32754, 500)

In [14]:
windows = np.expand_dims(windows, axis=-1)
windows.shape

(32754, 500, 1)

In [15]:
class TemporalAwareLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.05):
        super(TemporalAwareLSTM, self).__init__()
        
        # Multi-scale feature extraction to capture different temporal patterns
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(input_size, 16, kernel_size=k, padding=k//2) 
            for k in [3, 9, 15, 25]  # Multiple kernel sizes for different temporal scales
        ])
        
        # Enhanced input size with extracted features
        enhanced_input_size = input_size + 16*4  # Original + extracted features
        
        # Main LSTM with increased capacity
        self.rnn = nn.LSTM(enhanced_input_size, hidden_size, num_layers, 
                          batch_first=True, dropout=dropout, 
                          bidirectional=True, bias=True)
        
        # Sequential output layers
        self.fc1 = nn.Linear(hidden_size*2, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # Original signal
        orig_x = x
        
        # Extract multi-scale temporal features
        x_t = x.transpose(1, 2)  # [batch, channels, seq_len]
        conv_outputs = []
        
        for conv in self.conv_layers:
            conv_out = F.relu(conv(x_t))
            conv_outputs.append(conv_out.transpose(1, 2))  # Back to [batch, seq_len, features]
        
        # Concatenate original with extracted features
        enhanced_x = torch.cat([orig_x] + conv_outputs, dim=2)
        
        # Process through LSTM
        lstm_out, _ = self.rnn(enhanced_x)
        
        # Output through fully connected layers
        x = self.fc1(lstm_out)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)
        
        return out, x

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def load_model(model_path):
    """Load a saved model from the specified path."""
    # Initialize model with the same parameters as during training
    input_size = 1
    hidden_size = 64
    num_layers = 4
    output_size = 2
    dropout_rate = 0.05
    
    # Create model instance
    model = TemporalAwareLSTM(input_size, hidden_size, num_layers, output_size, dropout_rate)
    
    # Load the saved state
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'),weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Get the saved threshold for binary decisions
    threshold = checkpoint.get('threshold', 0.5)
    print(model_path,':',threshold)
    # Set model to evaluation mode
    model.eval()
    
    return model, threshold

In [17]:
from tqdm import tqdm   # nice progress bar, optional

def extract_embeddings(model, windows, take='center', batch_size=64, device='cpu'):
    """
    Parameters
    ----------
    model       : your loaded TemporalAwareLSTM, already in eval() mode
    windows     : np.ndarray or list, shape [N, window_len, 1]
    take        : 'center'  -> one 64-D vector at centre index of each window
                  'mean'    -> mean-pool over all timepoints in window
                  'full'    -> return full [N, window_len, 64] tensor
    batch_size  : batch size for inference
    device      : 'cpu' or 'cuda'

    Returns
    -------
    emb_np      : np.ndarray of shape
                  * [N, 64]   if take=='center' or 'mean'
                  * [N, window_len, 64] if take=='full'
    """

    model.to(device)
    all_emb = []

    # convert windows to torch tensor only once for efficiency
    windows_tensor = torch.as_tensor(windows, dtype=torch.float32)  # shape [N, L, 1]

    # simple batching
    n_samples = windows_tensor.shape[0]
    print(n_samples)
    for i in tqdm(range(0, n_samples, batch_size)):
        batch = windows_tensor[i:i+batch_size].to(device)  # [B, L, 1]

        with torch.no_grad():
            _, emb = model(batch)                          # emb shape [B, L, 64]

        # choose representation for each window
        if take == 'center':
            center_idx = emb.shape[1] // 2                 # L//2
            emb = emb[:, center_idx, :]                    # [B, 64]
        elif take == 'mean':
            emb = emb.mean(dim=1)                          # [B, 64]
        elif take == 'full':
            pass                                           # keep [B, L, 64]
        else:
            raise ValueError("take must be 'center', 'mean', or 'full'")

        all_emb.append(emb.cpu())

    emb_tensor = torch.cat(all_emb, dim=0)                 # [N, …]
    return emb_tensor.numpy()   

In [18]:
slow_model_path  ='/scratch/da3245/event_detection/models/events_slow_intermediate.pth'
fast_model_path = '/scratch/da3245/event_detection/models/events_fast_intermediate.pth'

In [19]:
model, threshold = load_model(fast_model_path)
model.eval()                                               # safety first
embeddings = extract_embeddings(
    model,
    windows,           # your sliding windows
    take='full',     # or 'mean', 'full'
    device='cuda'       # change to 'cuda' if GPU available
)

print(embeddings.shape)

/scratch/da3245/event_detection/models/events_fast_intermediate.pth : 0.5
32754


100%|██████████| 512/512 [00:06<00:00, 74.87it/s]


(32754, 500, 64)


In [20]:
embeddings.shape[0]

32754

In [22]:
import numpy as np
import os, json, pyarrow as pa, pyarrow.parquet as pq
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

ROOT      = "/scratch/da3245/neuro_app_data/v2025_08_07ft"
os.makedirs(ROOT, exist_ok=True)
traces = traces_np
labels_window = enc_lbls_np

# -----  1‑A  aggregate + PCA, if you haven't already  -----
emb_mean = embeddings.mean(1).astype("float32")        # (N, 64)
pca      = PCA(n_components=2, random_state=42)
pca_xy   = pca.fit_transform(emb_mean).astype("float32")

tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
tsne_xy = tsne.fit_transform(emb_mean).astype('float32')


# -----  1‑B  Export each array as a separate NPY file  -----
np.save(f"{ROOT}/traces.npy", traces_np.astype("float32"))         # (N, 500)
np.save(f"{ROOT}/label_seq.npy", labels_np.astype("uint8"))       # (N, 2, 500)
np.save(f"{ROOT}/encoded_labels.npy", enc_lbls_np.astype("uint8"))# (N,)
np.save(f"{ROOT}/emb_mean.npy", emb_mean)                          # (N, 64)
np.save(f"{ROOT}/pca_xy.npy", pca_xy)                              # (N, 2)
np.save(f"{ROOT}/tsne_xy.npy", tsne_xy)                              # (N, 2)
# Optionally skip origin_keys if it's an object array or convert to string/array if needed

print("✔ wrote", os.path.getsize(f'{ROOT}/traces.npy')/1e6, "MB (traces.npy)")

# -----  1‑C  minimal Parquet metadata (optional)  -----
meta = pa.table({
    "window_id" : pa.array(np.arange(len(traces))),
    "label_code": pa.array(labels_window.astype("uint8")),
    "pca_x"     : pa.array(pca_xy[:,0]),
    "pca_y"     : pa.array(pca_xy[:,1]),
})
pq.write_table(meta, f"{ROOT}/metadata.parquet", compression="zstd")

# -----  1‑D  manifest  -----
json.dump({
    "version": "v2025-07-24",
    "traces": "traces.npy",
    "label_seq": "label_seq.npy",
    "encoded_labels": "encoded_labels.npy",
    "emb_mean": "emb_mean.npy",
    "pca_xy": "pca_xy.npy",
    "tsne_xy":"tsne_xy.npy",
    "metadata": "metadata.parquet"
}, open(f"{ROOT}/manifest.json", "w"), indent=2)


✔ wrote 65.508128 MB (traces.npy)


In [21]:
import numpy as np
import os
import json
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import hdbscan

# --- Configuration ---
ROOT      = "/scratch/da3245/neuro_app_data/v2025_08_14ft"
os.makedirs(ROOT, exist_ok=True)
traces = traces_np
labels_window = enc_lbls_np
# Dummy data for demonstration
num_samples = 1000
timesteps = 500
embedding_dim = embeddings.shape[1]
num_labels = 4



# ----- Aggregate + Projections -----
emb_mean = embeddings.mean(1).astype("float32")
pca = PCA(n_components=2, random_state=42)
pca_xy = pca.fit_transform(emb_mean).astype("float32")
tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
tsne_xy = tsne.fit_transform(emb_mean).astype('float32')

# ----- HDBSCAN Clustering -----
clusterer = hdbscan.HDBSCAN(min_cluster_size=15, gen_min_span_tree=True)
cluster_labels = clusterer.fit_predict(emb_mean).astype("int16")

# ----- Export arrays as NPY files -----
np.save(f"{ROOT}/traces.npy", traces_np.astype("float32"))
np.save(f"{ROOT}/encoded_labels.npy", enc_lbls_np.astype("uint8"))
np.save(f"{ROOT}/cluster_labels.npy", cluster_labels)
np.save(f"{ROOT}/emb_mean.npy", emb_mean)

# ----- Enhanced Parquet metadata (without compression) -----
meta = pa.table({
    "window_id": pa.array(np.arange(len(traces_np))),
    "label_code": pa.array(enc_lbls_np.astype("uint8")),
    "cluster_code": pa.array(cluster_labels),
    "pca_x": pa.array(pca_xy[:, 0]),
    "pca_y": pa.array(pca_xy[:, 1]),
    "tsne_x": pa.array(tsne_xy[:, 0]),
    "tsne_y": pa.array(tsne_xy[:, 1]),
})
# Removed compression parameter for better compatibility
pq.write_table(meta, f"{ROOT}/metadata.parquet")

# ----- Create manifest -----
unique_clusters = np.unique(cluster_labels)
num_clusters = len(unique_clusters[unique_clusters != -1])
cluster_map = {str(i): f"Cluster {i}" for i in unique_clusters if i != -1}
if -1 in unique_clusters:
    cluster_map["-1"] = "Noise"

manifest = {
    "version": "v2025-08-14-tes",
    "description": (
        "This is a sample dataset for the Neuro-Visualizer application. "
        "It contains simulated neural time-series data segmented into 500-timestep windows. "
        "The data has been processed with PCA and t-SNE for dimensionality reduction, "
        "and clustered with HDBSCAN to identify distinct patterns."
    ),
    "num_samples": len(traces_np),
    "timesteps_per_sample": traces_np.shape[1],
    "embedding_dim": emb_mean.shape[1],
    "num_labels": num_labels,
    "num_clusters": num_clusters,
    "label_map": {"0": "Wake", "1": "N1", "2": "N2", "3": "REM"},
    "cluster_map": cluster_map,
    "files": {
        "traces": "traces.npy",
        "encoded_labels": "encoded_labels.npy",
        "cluster_labels": "cluster_labels.npy",
        "emb_mean": "emb_mean.npy",
        "metadata": "metadata.parquet"
    }
}

with open(f"{ROOT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("✔ Dataset manifest and uncompressed metadata written.")


/ext3/miniforge3/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/ext3/miniforge3/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


✔ Dataset manifest and uncompressed metadata written.
